# Train the XGBoost triage model on HI-Large (walk-forward evaluation)

This notebook trains and evaluates the triage model on IBM HI-Large. The features come from the Spark job in `spark/build_features_spark.py`, which writes them to S3 partitioned by day.

**Evaluation (walk-forward):**
- Horizon 7 days. A row for day `t` covers days `t-6..t`, and its label is 1 if the account is in a laundering transaction on days `t+1..t+7`.
- Four walk-forward folds, each with a 7-day test block. Each fold trains on earlier days and validates on a 7-day block before its test block. Purge gaps of 7 days separate train, validation and test.
- The hyperparameter grid is chosen on validation only. Each test block is scored once.
- A final block (the last 7 days) is held out after the folds. The final model is retrained on all development days and scored once. That final number is the result.

**Baseline:** the app's rule score, restricted to rules R1 to R4. R5 to R7 need the whole edge set, which the Spark job does not compute. The metrics state this.

**Conclusive:** a block is conclusive only if it has at least 200 laundering account-days. The output says so for every block.

Run this notebook from the `notebooks/` folder. The Spark feature job has to have finished first (see `README.md`).

## 1. Install pinned packages

The versions match the ones verified in the backend venv. If the install changes numpy or pandas on a running kernel, restart the kernel and run from section 2.

In [ ]:
%pip install -q "pandas==3.0.6" "numpy==2.5.4" "xgboost==3.4.1" "pyarrow==26.0.0" "scikit-learn==1.9.1" "boto3"

## 2. Configuration

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import time

import numpy as np
import pandas as pd

NB_DIR = Path.cwd()  # run from the notebooks/ folder
sys.path.insert(0, str(NB_DIR))

S3_BUCKET = "neu-datasets"
S3_FEATURES = f"s3://{S3_BUCKET}/features/hi_large"         # output of the Spark job
S3_CSV_KEY = "HI-Large_Trans.csv"                           # raw transactions (used for the leakage check)
S3_PATTERNS_KEY = "HI-Large_Patterns.txt"                   # optional, for per-typology recall

DATA_DIR = NB_DIR / "data"
FEATURES_DIR = NB_DIR / "features_large"                    # day partitions t=1, t=2, ...
MANIFEST_DIR = NB_DIR / "features_manifest"
MODELS_DIR = NB_DIR / "models"
EVAL_DIR = NB_DIR / "eval"
for folder in (DATA_DIR, FEATURES_DIR, MANIFEST_DIR, MODELS_DIR, EVAL_DIR):
    folder.mkdir(parents=True, exist_ok=True)

RUN_NAME = "triage_hi_large"
HORIZON = 7           # label horizon in days
TEST_DAYS = 7         # length of each test block
VAL_DAYS = 7          # length of each validation block (after a purge gap)
FOLDS = 4             # walk-forward folds, plus one final held-out block
BUDGET = 50           # placeholder: flagged account-days per day until the alert budget is confirmed
NEG_RATIO = 10        # training: 10 negatives per positive
VAL_NEG_RATIO = 100   # validation: sampled for early stopping only
UPLOAD_RESULTS = True # copy the model, manifest and results to s3://neu-datasets/triage/<RUN_NAME>/

## 3. Pull the Spark output and the manifest from S3

The features are synced once. Re-running this cell only copies what changed.

In [ ]:
def aws(*args):
    print("$ aws", " ".join(args), flush=True)
    subprocess.run(["aws", *args], check=True)

aws("s3", "sync", f"{S3_FEATURES}/features/", str(FEATURES_DIR), "--only-show-errors")
aws("s3", "cp", f"{S3_FEATURES}/manifest/features_manifest.json", str(MANIFEST_DIR / "features_manifest.json"),
    "--only-show-errors")

size_gb = sum(p.stat().st_size for p in FEATURES_DIR.rglob("*.parquet")) / 2**30
print(f"feature parts: {len(list(FEATURES_DIR.rglob('*.parquet')))} files, {size_gb:.2f} GiB")

patterns = DATA_DIR / S3_PATTERNS_KEY
if not patterns.exists():
    try:
        aws("s3", "cp", f"s3://{S3_BUCKET}/{S3_PATTERNS_KEY}", str(patterns), "--only-show-errors")
    except subprocess.CalledProcessError:
        print("no typology file in S3; per-typology recall will be skipped")
patterns = patterns if patterns.exists() else None
print("typology file:", patterns or "none")

## 4. Check the manifest against the configuration

In [ ]:
manifest = json.loads((MANIFEST_DIR / "features_manifest.json").read_text(encoding="utf-8"))
features = manifest["features"]
assert manifest["horizon"] == HORIZON, f"features were built with horizon {manifest['horizon']}, config says {HORIZON}"
max_t = manifest["max_day"] - HORIZON  # labels need HORIZON days after t
print(f"engine: {manifest.get('engine')}  rows: {manifest['rows']:,}  positives: {manifest['positives']:,}")
print(f"days: {manifest['day0']} + {manifest['max_day']} days  usable window days: 1..{max_t}")
print(f"features ({len(features)}): {features}")
print("baseline rules:", manifest.get("baseline_rules"))

## 5. Fold plan

Check the blocks before training. Each row is one block: the training days, the validation days, and the test days. A purge gap of `HORIZON` days sits between each pair.

In [ ]:
from triage_lib.triage_train import fold_plan

plans = fold_plan(max_t, HORIZON, FOLDS, TEST_DAYS, VAL_DAYS)
pd.DataFrame(plans)

## 6. Sanity check: future data must not change past features

This changes every transaction after a cutoff day in a sample of the raw CSV, rebuilds the features, and compares the rows up to the cutoff. The difference must be exactly 0. The check uses only the first ~100 MB of the CSV, so it runs in minutes.

In [ ]:
from triage_lib.alerts import load_csv
from triage_lib.triage_features import future_perturbation_diff

sample_csv = DATA_DIR / "HI-Large_sample.csv"
if not sample_csv.exists():
    aws("s3api", "get-object", "--bucket", S3_BUCKET, "--key", S3_CSV_KEY, "--range", "bytes=0-100000000",
        str(sample_csv.with_suffix(".part")))
    raw = sample_csv.with_suffix(".part").read_bytes()
    sample_csv.write_bytes(raw[:raw.rfind(b"\n") + 1])  # drop the cut-off last line
    sample_csv.with_suffix(".part").unlink()

sample = load_csv(str(sample_csv))
print(f"sample: {len(sample):,} transactions, {sample['ts'].min()} to {sample['ts'].max()}")
diff = future_perturbation_diff(sample, horizon=HORIZON, cutoff_day=3)
print("max feature difference for rows up to day 3:", diff)
assert diff == 0.0, "future data changes past features: leakage, do not train"

## 7. Walk-forward evaluation and the final model

Folds run one after another. Each fold's grid is chosen on its validation block. Then the final model is trained with the grid point with the best mean validation score, and scored once on the final block. Runtime is the largest cost here: 12 grid points per fold, plus the final retrain.

In [ ]:
from triage_lib.triage_train import walk_forward

t0 = time.time()
result = walk_forward(
    path=FEATURES_DIR,
    features=features,
    horizon=HORIZON,
    max_t=max_t,
    folds=FOLDS,
    test_days=TEST_DAYS,
    val_days=VAL_DAYS,
    budget=BUDGET,
    out_dir=EVAL_DIR / RUN_NAME,
    model_dir=MODELS_DIR / RUN_NAME,
    manifest_path=MANIFEST_DIR / "features_manifest.json",
    neg_ratio=NEG_RATIO,
    day0=pd.Timestamp(manifest["day0"]),
    patterns_path=patterns,
    val_neg_ratio=VAL_NEG_RATIO,
)
print(f"walk-forward finished in {(time.time() - t0) / 60:.1f} min")
print("selected parameters:", result["selected_params"])
print("final rounds:", result["final"]["rounds"])

## 8. Results

One row per block and scorer, at the budget in `BUDGET`. `conclusive = False` means the block has fewer than 200 positives, so its numbers are not reliable on their own. The summary shows the spread across the four folds. The final row is the result.

In [ ]:
rows = []
for block in result["folds"] + [result["final"]]:
    for scorer in ("model", "rules_app"):
        m = block[scorer]
        rows.append({
            "block": block["name"], "scorer": "XGBoost" if scorer == "model" else "app rule score (R1-R4)",
            "test_positives": block["positives"]["test"], "conclusive": block["conclusive"],
            "precision": m["precision_at_budget"], "recall": m["recall_at_budget"], "lift": m["lift"],
            "pr_auc": m["pr_auc"], "tp": m["tp"], "fp": m["fp"], "fn": m["fn"],
        })
results_table = pd.DataFrame(rows)
display(results_table)

summary = pd.DataFrame({
    scorer: {k: v["mean"] for k, v in result["summary"][key].items()}
    for scorer, key in (("XGBoost", "model"), ("app rule score (R1-R4)", "rules_app"))
})
print("fold mean across the 4 folds:")
display(summary)
print("folds that are conclusive:", result["summary"]["folds_conclusive"], "of", FOLDS)

## 9. Sanity check: shuffled labels

The labels are permuted within each day and the final model's PR-AUC is recomputed. It should drop to the base rate. If it stays well above, something in the features or labels leaks.

In [ ]:
import xgboost as xgb
from triage_lib.triage_train import label_shuffle_check, score_days

booster = xgb.Booster()
booster.load_model(str(MODELS_DIR / RUN_NAME / "model.json"))
final_plan = result["plans"][-1]
scored = score_days(FEATURES_DIR, final_plan["test_start"], final_plan["test_end"], features, booster,
                    iteration_range=(0, result["final"]["rounds"]))
print(label_shuffle_check(scored, "score"))
print("real final PR-AUC:", result["final"]["model"]["pr_auc"])

## 10. Per-typology recall (final block)

Only if the patterns file was found. The recall for each typology is the share of its laundering account-days that were flagged at the budget.

In [ ]:
typology = result["final"].get("typology_recall_test") or {}
if typology:
    display(pd.DataFrame(typology).T)
else:
    print("per-typology recall skipped (no HI-Large_Patterns.txt in S3)")

## 11. Save and upload

The model, the feature manifest and the library versions are saved together, which is what the batch scorer needs. The results go to `eval/` and to S3.

In [ ]:
import boto3

print("model folder:", MODELS_DIR / RUN_NAME, sorted(p.name for p in (MODELS_DIR / RUN_NAME).iterdir()))
print("versions:", json.loads((MODELS_DIR / RUN_NAME / "versions.json").read_text()))
print("results:", EVAL_DIR / RUN_NAME / "walk_forward.json")

if UPLOAD_RESULTS:
    s3 = boto3.client("s3")
    uploads = [(MODELS_DIR / RUN_NAME / name, f"triage/{RUN_NAME}/model/{name}")
               for name in ("model.json", "features_manifest.json", "versions.json")]
    uploads.append((EVAL_DIR / RUN_NAME / "walk_forward.json", f"triage/{RUN_NAME}/walk_forward.json"))
    for local, key in uploads:
        s3.upload_file(str(local), S3_BUCKET, key)
        print(f"uploaded s3://{S3_BUCKET}/{key}")

## Reading the result

- **Positive result:** the XGBoost precision and recall at the budget beat the app rule score on the final block, and the folds agree in direction. Only then consider switching triage (`TRIAGE_MODEL=xgb`).
- **Not conclusive:** any block with fewer than 200 positives. Its numbers are shown but should not decide anything.
- **Limits:** HI-Large is synthetic, from the same generator as HI-Small, so real-world performance will be lower. The baseline is R1 to R4 only. The budget of 50 per day is a placeholder.